In [ ]:
# Extract FLPs from CSD
from ccdc.io import CrystalReader

def extract_flp_smiles():
    """ Extracts FLP molecules from CSD using their refcodes """
    csd_reader = CrystalReader('CSD')
    flp_refcodes = ["FLP1", "FLP2", "FLP3"]  # Replace with real FLP refcodes
    smiles_list = []
    
    for refcode in flp_refcodes:
        try:
            mol = csd_reader.molecule(refcode)
            smiles_list.append(mol.smiles)
        except:
            print(f"Could not retrieve {refcode}")

    return smiles_list

flp_smiles = extract_flp_smiles()
print(f"Extracted FLPs: {flp_smiles}")

In [ ]:
# CSD surface charge calculator
from ccdc.io import CrystalReader
from surface_charge_calculator import SurfaceChargeController

def compute_surface_charge(flp_refcode):
    """ Computes the surface charge for a given FLP structure """
    csd_reader = CrystalReader('CSD')
    structure = csd_reader.molecule(flp_refcode)

    charge_calculator = SurfaceChargeController(structure, hkl_and_offsets=[(1,0,0,0.1)])
    charge_calculator.calculate_surface_charge()

    return charge_calculator.surface_atom_charges[0]  # Return total surface charge

flp_refcode = "FLP1"
charge = compute_surface_charge(flp_refcode)
print(f"Surface charge for {flp_refcode}: {charge}")

In [ ]:
# CSD Hydrogen Bond Scores
from ccdc.io import CrystalReader
from hydrogen_bond_propensity_report import PropensityCalc

def compute_hydrogen_bonding(flp_refcode):
    """ Computes hydrogen bonding propensity for a given FLP """
    csd_reader = CrystalReader('CSD')
    crystal = csd_reader.crystal(flp_refcode)

    hbp_calculator = PropensityCalc(crystal)
    propensities = hbp_calculator.calculate()

    return sum([p.propensity for p in propensities])  # Sum of hydrogen bond scores

flp_refcode = "FLP1"
hbond_score = compute_hydrogen_bonding(flp_refcode)
print(f"Hydrogen bond propensity for {flp_refcode}: {hbond_score}")

In [ ]:
# CSD Semiconductor Properties Evaluation
from show_semiconductor_properties import write_descs_report

def compute_semiconductor_properties():
    """ Runs semiconductor property evaluation for FLPs """
    write_descs_report()  # Generates semiconductor property report

compute_semiconductor_properties()

In [ ]:
def compute_flp_reward(smiles):
    """ Computes reward score for a generated FLP molecule """
    flp_refcode = "FLP1"  # Use extracted FLP
    charge = compute_surface_charge(flp_refcode)
    hbond_score = compute_hydrogen_bonding(flp_refcode)
    
    return charge * 0.5 + hbond_score * 0.5  # Weighted reward score

# Update Generator Loss Function
def rl_guided_loss(generator, discriminator, batch_size=64):
    """ Custom RL loss function for Generator """
    z = torch.randn(batch_size, 128)
    fake_samples = generator(z).detach().numpy()
    
    # Convert generated molecules to SMILES (mock data)
    smiles_list = ["CCOC(=O)C1=CC=CC=C1", "CCC(=O)OCC"]

    # Compute RL Rewards
    rewards = torch.tensor([compute_flp_reward(smi) for smi in smiles_list], dtype=torch.float32)

    # Combine WGAN-GP loss with RL Reward
    loss_G = -torch.mean(discriminator(generator(z))) + torch.mean(rewards)
    return loss_G